# 第2讲：工程数据表达与Python／Jupyter预备（学生操作）

> 目标环境：Windows 64位、Python 3.12.10、`D:\EngineeringBigData\venv`、JupyterLab。课程环境已由管理员课前配置，课堂不安装软件包。

## 当堂练习（约15分钟）

1. 从头运行到缺失检查，核对18行×12列以及两个缺失字段；
2. 把`FOCUS_RESULT`设为“复核”，得到4条记录；
3. 把`GROUP_FIELD`设为`result`，得到通过11、复核4、整改3；
4. 生成`outputs/分组统计图.png`，再执行“Restart Kernel and Run All”；
5. 保存Notebook，并确认`outputs/自检结果.txt`显示PASS。

保持Notebook、`data`和`outputs`的相对位置。遇到报错时记录电脑编号、失败单元格和报错信息，不在课堂上自行安装软件包。


## 1. 环境与路径

In [ ]:
from __future__ import annotations

import platform
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TARGET_PYTHON = "3.12.10"

print({
    "target_python": TARGET_PYTHON,
    "runtime_python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
})

In [ ]:
DATA_PATH = Path("data/构件检查示例数据.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"],
    "axes.unicode_minus": False,
    "figure.dpi": 120,
})

print("数据路径：", DATA_PATH)
print("数据文件存在：", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError("找不到CSV。请在Windows文件资源管理器中完整解压素材包，并保留Notebook与data目录的相对位置。")

## 2. 读取CSV并查看前5行

示例数据每行代表**一次检查事件**。inspection_id标识检查事件；component_id引用被检查构件。

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"读取完成：{df.shape[0]}行，{df.shape[1]}列")
display(df.head())

## 3. 查看行列、字段和当前存储类型

dtypes显示当前存储类型；数据字典说明变量的工程含义、测量尺度和分析角色。

In [ ]:
print("shape:", df.shape)
print("columns:", list(df.columns))
print("\ndtypes:")
print(df.dtypes)
print("\ninfo:")
df.info()

## 4. 检查缺失

本节统计缺失字段和数量。删除或填补缺失时，需要结合字段含义、任务和处理记录。

In [ ]:
missing = df.isna().sum()
missing_nonzero = missing[missing > 0]
print("存在缺失的字段：")
print(missing_nonzero)

## 5. 条件筛选

把FOCUS_RESULT改为“复核”并运行。课后复做时再改为“整改”。

In [ ]:
FOCUS_RESULT = "复核"  # 课堂改这里；可选：通过、复核、整改
focus_rows = df.loc[df["result"] == FOCUS_RESULT].copy()

print(f"result = {FOCUS_RESULT}：{len(focus_rows)}条检查事件")
display(focus_rows[[
    "inspection_id", "component_id", "component_type",
    "zone", "inspection_time", "result"
]])

## 6. 分组计数

GROUP_FIELD可选择result、component_type或zone。原表一行一次检查，因此下列数量是**检查事件数**。

In [ ]:
GROUP_FIELD = "result"  # 课堂可改为component_type；课后改为zone
ALLOWED_GROUP_FIELDS = {"result", "component_type", "zone"}

if GROUP_FIELD not in ALLOWED_GROUP_FIELDS:
    raise ValueError(f"GROUP_FIELD应从以下字段中选择：{sorted(ALLOWED_GROUP_FIELDS)}")

summary = (
    df.groupby(GROUP_FIELD, dropna=False)
      .size()
      .reset_index(name="inspection_count")
      .sort_values("inspection_count", ascending=False)
      .reset_index(drop=True)
)

display(summary)

## 7. 绘图并保存

柱高表示当前18条示例记录中的检查事件数量。

In [ ]:
FIELD_LABELS = {
    "result": "检查结果",
    "component_type": "构件类型",
    "zone": "区域",
}
OUTPUT_PATH = OUTPUT_DIR / "分组统计图.png"

ax = summary.plot.bar(
    x=GROUP_FIELD,
    y="inspection_count",
    legend=False,
    color="#2F5D8A",
    figsize=(7.0, 4.2),
)
ax.set_xlabel(FIELD_LABELS[GROUP_FIELD])
ax.set_ylabel("检查事件数")
ax.set_title(f"按{FIELD_LABELS[GROUP_FIELD]}统计的检查事件数量")
ax.tick_params(axis="x", rotation=0)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(OUTPUT_PATH, dpi=160, bbox_inches="tight")

print("图已保存：", OUTPUT_PATH)

## 8. 环境与结果自检

运行后应显示PASS，并在outputs生成自检结果.txt。

In [ ]:
EXPECTED_COLUMNS = [
    "inspection_id", "component_id", "component_type", "zone",
    "inspection_time", "recorded_at", "result", "severity_level",
    "defect_count", "temperature_c", "inspector", "data_version",
]
EXPECTED_RESULT_COUNTS = {"通过": 11, "复核": 4, "整改": 3}

assert DATA_PATH.exists()
assert df.shape == (18, 12), df.shape
assert list(df.columns) == EXPECTED_COLUMNS
assert df["inspection_id"].notna().all()
assert df["inspection_id"].is_unique
assert df["component_id"].nunique() == 14
assert df["result"].value_counts().to_dict() == EXPECTED_RESULT_COUNTS
assert int(missing["inspector"]) == 1
assert int(missing["temperature_c"]) == 1
assert len(focus_rows) == int((df["result"] == FOCUS_RESULT).sum())
assert int(summary["inspection_count"].sum()) == len(df)
assert OUTPUT_PATH.exists()

check_text = "\n".join([
    "PASS: 第2讲数据、路径与分析结果自检通过",
    f"Target Python: {TARGET_PYTHON}",
    f"Runtime Python: {platform.python_version()}",
    f"pandas: {pd.__version__}",
    f"data shape: {df.shape}",
    f"focus result: {FOCUS_RESULT}, rows: {len(focus_rows)}",
    f"group field: {GROUP_FIELD}",
    f"figure: {OUTPUT_PATH}",
    "Data status: 18条教学合成检查记录",
])

CHECK_PATH = OUTPUT_DIR / "自检结果.txt"
CHECK_PATH.write_text(check_text, encoding="utf-8")
print(check_text)
print("自检记录已保存：", CHECK_PATH)

## 当堂记录与课后复做

请在此Markdown单元中补充：

1. 两个存在缺失的字段及缺失数量；
2. 为什么`inspection_time`显示为`object`，仍不能把它简单理解为普通文本；
3. 为什么按`result`统计的三组数量相加为18，而不是构件数量。

课后可把`FOCUS_RESULT`改为“整改”，把`GROUP_FIELD`改为`zone`，重新运行并比较新图。
